# 0.Library

In [32]:
import pandas as pd
from pathlib import Path

import importlib
import sys
sys.path.append('../') 
from features import data_utils as du
from features import data_pipeline as dp

# Reload
importlib.reload(du)
importlib.reload(dp)

# Constants and paths
parent_folder = Path("../..") # go 2 folder up: "../.."
data_folder = parent_folder / "data"
input_path_of_json = data_folder / "patients_data_log.json"
df_data_log = pd.read_json(input_path_of_json)

patient_ids = df_data_log["PatientID"].to_list()
patients_data_folder = data_folder / "PatientsData"

# feature vectors

In [2]:
# feature vectors
features_dict = {
    "section_id": None,
    "section_name": None,
    "section_total_time": None,
}

# statistical features
hover_dict = {
    "total_hover_count": None,
    "total_hover_duration": None,
    "mean_hover_duration": None,
    "max_hover_duration": None,
    "median_hover_duration": None,
    "std_hover_duration": None,
    "cv_hover_duration": None, # std_hover_duration / mean_hover_duration
    "hover_intensity": None
}

press_dict = {
    "total_press_count": None,
    "total_press_duration": None,
    "mean_press_duration": None,
    "max_press_duration": None,
    "median_press_duration": None,
    "std_press_duration": None,
    "press_intensity": None # Interaction intensity
}

reading_time_dict = {
    "total_reading_time_duration": None,
    "mean_reading_time_duration": None,
    "max_reading_time_duration": None,
    "median_reading_time_duration": None,
    "std_reading_time_duration": None,
    "reading_time_intensity" : None
}

# behavioral features
behavior_dict = {
    "hover_vs_active_interaction_ratio": None,
    "hover_vs_reading_time_ratio": None,
    "interaction_fraction": None,
    "decision_latency": None,  
    "clicks_per_second": None,
    "hovers_per_click": None
}

# Temporal Features
temporal_dict = {
    "time_before_first_press": None,
    "time_before_first_hover": None
}

# get clean path of one patient (all csv files)

In [3]:
# get clean path of one patient (all csv files)
patient_id = patient_ids[0]
patient_folder_path = du.find_patient_folder(patients_data_folder=patients_data_folder,
                                               patient_id=patient_id)
                                               
path_to_check = patient_folder_path / "clean_data"
clean_csv_files = du.find_csv_file(patient_folder_path / "clean_data")
clean_csv_files

['cleaned_20260502_113117_01_Tutorial_events.csv',
 'cleaned_20260502_113117_01_Tutorial_trajectory.csv',
 'cleaned_20260502_113117_02_ObjectRecognition_events.csv',
 'cleaned_20260502_113117_02_ObjectRecognition_trajectory.csv',
 'cleaned_20260502_113117_03_Visuospatial_events.csv',
 'cleaned_20260502_113117_03_Visuospatial_trajectory.csv',
 'cleaned_20260502_113117_04_Memory_events.csv',
 'cleaned_20260502_113117_04_Memory_trajectory.csv']

# working with 01_Tutorial_events

In [4]:
# start with the first csv file
clean_df = pd.read_csv(path_to_check / clean_csv_files[0])

In [5]:
clean_df.head()

,Timestamp,PhaseTime_s,Section,EventType,ObjectName,Hand,IsCorrect,Duration_s,Distance_m,Details,Activity_Log
0,2026-05-02 11:31:17.628,0.000,Default,PHASE_START,NaN,NaN,NaN,0.0,0.0,Phase=01_Tutorial,"['Section=Default', 'SessionID=20260502_113117..."
1,2026-05-02 11:31:17.630,0.000,Default,PANEL_SHOWN,WelcomePanel,NaN,NaN,0.0,0.0,Section=Default,['Panel=WelcomePanel']
2,2026-05-02 11:31:22.971,0.000,Default,BUTTON_HOVER_START,NaN,NaN,NaN,0.0,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...
3,2026-05-02 11:31:23.918,0.952,Default,BUTTON_HOVER_END,NaN,NaN,NaN,0.0,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...
4,2026-05-02 11:31:26.072,3.106,Default,BUTTON_HOVER_START,NaN,NaN,NaN,0.0,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...


# working with one section

In [33]:
section_names = clean_df['Section'].unique()
section_df = du.filter_on_section(clean_df, section_names[0])
section_df.head()

,Timestamp,PhaseTime_s,Section,EventType,ObjectName,Hand,IsCorrect,Duration_s,Distance_m,Details,Activity_Log
0,2026-05-02 11:31:17.628,0.000,Default,PHASE_START,NaN,NaN,NaN,0.0,0.0,Phase=01_Tutorial,"['Section=Default', 'SessionID=20260502_113117..."
1,2026-05-02 11:31:17.630,0.000,Default,PANEL_SHOWN,WelcomePanel,NaN,NaN,0.0,0.0,Section=Default,['Panel=WelcomePanel']
2,2026-05-02 11:31:22.971,0.000,Default,BUTTON_HOVER_START,NaN,NaN,NaN,0.0,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...
3,2026-05-02 11:31:23.918,0.952,Default,BUTTON_HOVER_END,NaN,NaN,NaN,0.0,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...
4,2026-05-02 11:31:26.072,3.106,Default,BUTTON_HOVER_START,NaN,NaN,NaN,0.0,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...


# extract events

In [7]:
events = section_df['EventType'].unique()
events

array(['PHASE_START', 'PANEL_SHOWN', 'BUTTON_HOVER_START',
       'BUTTON_HOVER_END', 'BUTTON_PRESSED', 'BUTTON_RELEASED',
       'PANEL_DISMISSED', 'SECTION_END'], dtype=object)

# Bulding feature vector for one section

## extract section duration

In [34]:
# extract section duration
section_total_time = du.extract_section_duration(section_df)
section_total_time

17.23

## calculation for Hover dictionary

In [35]:
# hover count
du.fill_hover_dict_Default_section(section_df, hover_dict, section_total_time)

{'total_hover_count': np.float64(28.0),
 'total_hover_duration': np.float64(5.52),
 'mean_hover_duration': np.float64(0.79),
 'max_hover_duration': np.float64(2.29),
 'median_hover_duration': np.float64(0.54),
 'std_hover_duration': np.float64(0.72),
 'cv_hover_duration': np.float64(0.91),
 'hover_intensity': np.float64(0.32)}

## calculation for Press dictionary

In [ ]:
# press count (assumed that for each button pressed, there is a button released too
filtered_df_button_pressed = du.filter_by_string_contains(section_df, 'EventType', 'BUTTON_PRESSED')
total_press_count = len(filtered_df_button_pressed)  # Assuming each press has a corresponding release
press_dict["total_press_count"] = total_press_count

# press duration
filtered_df = du.filter_by_string_contains(section_df, 'Activity_Log', 'PressDuration')
press_duration_series = du.extract_metric_from_section(filtered_df, du.extract_press_duration)

press_dict["total_press_duration"], press_dict["mean_press_duration"], press_dict["max_press_duration"], press_dict["median_press_duration"], press_dict["std_press_duration"] = du.calculate_metric_stats(press_duration_series, is_counting=False)

# press_intensity
press_dict["press_intensity"] = du.ratio_calculation(press_dict["total_press_duration"], section_total_time)

press_dict

{'total_press_count': 1,
 'total_press_duration': np.float64(0.22),
 'mean_press_duration': np.float64(0.22),
 'max_press_duration': np.float64(0.22),
 'median_press_duration': np.float64(0.22),
 'std_press_duration': np.float64(nan),
 'press_intensity': np.float64(0.01)}

## calculation for Reading Time dictionary

In [ ]:
# reading time duration
filtered_df = du.filter_by_string_contains(section_df, 'Activity_Log', 'ReadingTime')
reading_time_series = du.extract_metric_from_section(filtered_df, du.extract_reading_time_duration)

# calculation for reading time duration
reading_time_dict["total_reading_time_duration"], reading_time_dict["mean_reading_time_duration"], reading_time_dict["max_reading_time_duration"], reading_time_dict["median_reading_time_duration"], reading_time_dict["std_reading_time_duration"] = du.calculate_metric_stats(reading_time_series, is_counting=False)

# reading time intensity
reading_time_dict["reading_time_intensity"] = du.ratio_calculation(reading_time_dict["total_reading_time_duration"], section_total_time)

reading_time_dict

{'total_reading_time_duration': np.float64(17.23),
 'mean_reading_time_duration': np.float64(17.23),
 'max_reading_time_duration': np.float64(17.23),
 'median_reading_time_duration': np.float64(17.23),
 'std_reading_time_duration': np.float64(nan),
 'reading_time_intensity': np.float64(1.0)}

## calculation for Behavior dictionary

In [ ]:
behavior_dict = {
    "hover_vs_active_interaction_ratio": None,
    "hover_vs_reading_time_ratio": None,
    "interaction_fraction": None,
    "decision_latency": None,  
    "clicks_per_second": None,
    "hovers_per_click": None
}

In [ ]:
behavior_dict["hover_vs_active_interaction_ratio"] = du.ratio_calculation(hover_dict["total_hover_duration"], press_dict["total_press_duration"])
behavior_dict["hover_vs_reading_time_ratio"] = du.ratio_calculation(hover_dict["total_hover_duration"], reading_time_dict["total_reading_time_duration"])

behavior_dict["interaction_fraction"] = du.ratio_calculation(hover_dict["total_hover_duration"] + press_dict["total_press_duration"], section_total_time)

behavior_dict["decision_latency"] = du.calculate_decision_latency(first_hover_time=du.extract_first_time_hover(section_df), first_press_time=du.extract_first_time_press(section_df))

behavior_dict["clicks_per_second"] = du.ratio_calculation(press_dict["total_press_count"], section_total_time)
behavior_dict["hovers_per_click"] = du.ratio_calculation(hover_dict["total_hover_count"], press_dict["total_press_count"])

behavior_dict

{'hover_vs_active_interaction_ratio': np.float64(25.09),
 'hover_vs_reading_time_ratio': np.float64(0.32),
 'interaction_fraction': np.float64(0.33),
 'decision_latency': np.float64(17.013),
 'clicks_per_second': 0.06,
 'hovers_per_click': np.float64(28.0)}

## calculation for Temporal dictionary

In [ ]:
temporal_dict['time_before_first_press'] = du.extract_first_time_press(section_df)
temporal_dict['time_before_first_hover'] = du.extract_first_time_hover(section_df)

temporal_dict

{'time_before_first_press': np.float64(17.013),
 'time_before_first_hover': np.float64(0.0)}

# Final feature vector

In [ ]:
features_dict["section_id"] = 1
features_dict["section_name"] = section_names[0]
features_dict["section_total_time"] = section_total_time

features_dict.update(hover_dict)
features_dict.update(press_dict)
features_dict.update(reading_time_dict)
features_dict.update(behavior_dict)
features_dict.update(temporal_dict)

df = pd.DataFrame([features_dict])
df

,section_id,section_name,section_total_time,total_hover_count,total_hover_duration,mean_hover_duration,max_hover_duration,median_hover_duration,std_hover_duration,cv_hover_duration,...,std_reading_time_duration,reading_time_intensity,hover_vs_active_interaction_ratio,hover_vs_reading_time_ratio,interaction_fraction,decision_latency,clicks_per_second,hovers_per_click,time_before_first_press,time_before_first_hover
0,1,Default,17.23,28,5.52,0.79,2.29,0.54,0.72,0.91,...,NaN,1.0,25.09,0.32,0.33,17.013,0.06,28.0,17.013,0.0


In [ ]:
dp.creating_feature_vector_section_default(section_df,
                                           hover_dict,
                                            press_dict,
                                            reading_time_dict,
                                            behavior_dict,
                                            temporal_dict)

,section_id,section_name,section_total_time,total_hover_count,total_hover_duration,mean_hover_duration,max_hover_duration,median_hover_duration,std_hover_duration,cv_hover_duration,...,std_reading_time_duration,reading_time_intensity,hover_vs_active_interaction_ratio,hover_vs_reading_time_ratio,interaction_fraction,decision_latency,clicks_per_second,hovers_per_click,time_before_first_press,time_before_first_hover
0,1,Default,17.23,28,5.52,0.79,2.29,0.54,0.72,0.91,...,NaN,1.0,25.09,0.32,0.33,17.013,0.06,28.0,17.013,0.0


# working with section 2 == ButtonSelection

In [12]:
section_names = clean_df['Section'].unique()
section_df = du.filter_on_section(clean_df, section_names[1])
section_df.head()

,Timestamp,PhaseTime_s,Section,EventType,ObjectName,Hand,IsCorrect,Duration_s,Distance_m,Details,Activity_Log
20,2026-05-02 11:31:40.208,17.235,ButtonSelection,SECTION_START,NaN,NaN,NaN,0.0,0.0,Section=ButtonSelection,['PhaseTime=17.23s']
21,2026-05-02 11:31:40.209,17.235,ButtonSelection,PANEL_SHOWN,ButtonInstructionPanel,NaN,NaN,0.0,0.0,Section=ButtonSelection,['Panel=ButtonInstructionPanel']
22,2026-05-02 11:31:40.209,17.235,ButtonSelection,BUTTON_CLICKED,NaN,NaN,NaN,0.0,0.0,Section=ButtonSelection,['Button=StartTutorialButton[TutorialWelcome]'...
23,2026-05-02 11:31:44.409,21.443,ButtonSelection,BUTTON_HOVER_START,NaN,NaN,NaN,0.0,0.0,Section=ButtonSelection,['Button=PracticeButton1[TutorialButtonSelecti...
24,2026-05-02 11:31:45.506,22.540,ButtonSelection,BUTTON_PRESSED,NaN,NaN,NaN,0.0,0.0,Section=ButtonSelection,['Button=PracticeButton1[TutorialButtonSelecti...


In [11]:
# feature vectors
features_dict2 = {
    "section_id": None,
    "section_name": None,
    "section_total_time": None,
}

# statistical features
hover_dict2 = {
    "total_hover_count": None,
    "total_hover_duration": None,
    "mean_hover_duration": None,
    "max_hover_duration": None,
    "median_hover_duration": None,
    "std_hover_duration": None,
    "cv_hover_duration": None, # std_hover_duration / mean_hover_duration
    "hover_intensity": None
}

press_dict2 = {
    "total_press_count": None,
    "total_press_duration": None,
    "mean_press_duration": None,
    "max_press_duration": None,
    "median_press_duration": None,
    "std_press_duration": None,
    "press_intensity": None # Interaction intensity
}

reading_time_dict2 = {
    "total_reading_time_duration": None,
    "mean_reading_time_duration": None,
    "max_reading_time_duration": None,
    "median_reading_time_duration": None,
    "std_reading_time_duration": None,
    "reading_time_intensity" : None
}

# behavioral features
behavior_dict2 = {
    "hover_vs_active_interaction_ratio": None,
    "hover_vs_reading_time_ratio": None,
    "interaction_fraction": None,
    "decision_latency": None,  
    "clicks_per_second": None,
    "hovers_per_click": None
}

# Temporal Features
temporal_dict2 = {
    "time_before_first_press": None,
    "time_before_first_hover": None
}

In [ ]:
# extract section duration
section_total_time = du.extract_section_duration(section_df)
section_total_time

7.8

# Building feature vector 

## hover dictionary

In [31]:
du.fill_hover_dict_ButtonSelection_section(section_df, hover_dict2, section_total_time)

{'total_hover_count': np.float64(13.0),
 'total_hover_duration': np.float64(5.64),
 'mean_hover_duration': np.float64(0.94),
 'max_hover_duration': np.float64(1.51),
 'median_hover_duration': np.float64(0.99),
 'std_hover_duration': np.float64(0.49),
 'cv_hover_duration': np.float64(0.52),
 'hover_intensity': np.float64(0.72)}

## Press dictionary 

In [ ]:
# press count (assumed that for each button pressed, there is a button released too
filtered_df_button_pressed = (section_df['EventType'] == 'BUTTON_PRESSED') | section_df['EventType'] == 'BUTTON_PRESSED' | section_df['EventType'] == 'BUTTON_CLICKED'
filtered_df_button_pressed = du.filter_by_string_contains(section_df, 'EventType', 'BUTTON_PRESSED')
total_press_count = len(filtered_df_button_pressed)  # Assuming each press has a corresponding release
press_dict["total_press_count"] = total_press_count

# press duration
filtered_df = du.filter_by_string_contains(section_df, 'Activity_Log', 'PressDuration')
press_duration_series = du.extract_metric_from_section(filtered_df, du.extract_press_duration)

press_dict["total_press_duration"], press_dict["mean_press_duration"], press_dict["max_press_duration"], press_dict["median_press_duration"], press_dict["std_press_duration"] = du.calculate_metric_stats(press_duration_series, is_counting=False)

# press_intensity
press_dict["press_intensity"] = du.ratio_calculation(press_dict["total_press_duration"], section_total_time)

press_dict